# IMPORT

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

# DEFINE FOLDERS

In [2]:
STATION_FEATURE_FOLDER = Path("station_features")
DATASET_INPUT_FOLDER = Path("correlated_data_csv")

folders = [DATASET_INPUT_FOLDER, STATION_FEATURE_FOLDER]
for folder in folders:
    folder.mkdir(parents=True, exist_ok=True)

# LOAD DATA

In [3]:
df = pd.read_csv(f"{DATASET_INPUT_FOLDER}/clean_bike_weather_data.csv")

print("Rows:", df.shape[0])
print("Stations:", df["stationcode"].nunique())

Rows: 4763007
Stations: 1474


# NORMALIZE USAGE

In [4]:
df["usage_ratio"] = df["total_bikes_now"] / df["capacity"]
df["usage_ratio"] = df["usage_ratio"].clip(0, 1)

# DEFINA FEATURE EXTRACTION PER STATION

In [5]:
def extract_station_features(group):
    features = {}

    # --- Basic usage ---
    features["mean_usage"] = group["usage_ratio"].mean()
    features["std_usage"] = group["usage_ratio"].std()
    features["min_usage"] = group["usage_ratio"].min()
    features["max_usage"] = group["usage_ratio"].max()

    # --- Temporal behavior ---
    features["mean_weekend"] = group[group["is_weekend"] == 1]["usage_ratio"].mean()
    features["mean_weekday"] = group[group["is_weekend"] == 0]["usage_ratio"].mean()

    features["day_usage"] = group[group["weather_is_day"] == 1]["usage_ratio"].mean()
    features["night_usage"] = group[group["weather_is_day"] == 0]["usage_ratio"].mean()

    # --- Daily amplitude ---
    hourly_mean = group.groupby("hour")["usage_ratio"].mean()
    features["daily_amplitude"] = hourly_mean.max() - hourly_mean.min()

    # --- Weather sensitivity (CORRELATIONS) ---
    for col in [
        "weather_temperature_2m",
        "weather_wind_speed_10m",
        "weather_wind_direction_10m",
        "weather_weather_code"
    ]:
        if group[col].nunique() > 1:
            features[f"corr_{col}"] = group["usage_ratio"].corr(group[col])
        else:
            features[f"corr_{col}"] = 0.0

    # --- Spatial info (kept for later analysis) ---
    features["latitude"] = group["latitude"].iloc[0]
    features["longitude"] = group["longitude"].iloc[0]
    features["capacity"] = group["capacity"].iloc[0]

    return pd.Series(features)


# BUILD STATION FEATURE TABLE

In [6]:
station_features = (
    df.groupby("stationcode", sort=False)
    .apply(extract_station_features)
    .reset_index()
)

print("Station feature table shape:", station_features.shape)
station_features.head()

Station feature table shape: (1474, 17)


C:\Users\emirh\AppData\Local\Temp\ipykernel_36392\1111460357.py:3: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(extract_station_features)


,stationcode,mean_usage,std_usage,min_usage,max_usage,mean_weekend,mean_weekday,day_usage,night_usage,daily_amplitude,corr_weather_temperature_2m,corr_weather_wind_speed_10m,corr_weather_wind_direction_10m,corr_weather_weather_code,latitude,longitude,capacity
0,14111.0,0.214797,0.174374,0.000000,0.96,0.182422,0.227075,0.320265,0.152220,0.311798,0.036309,0.030049,-0.038988,0.011178,48.837526,2.336035,25.0
1,32017.0,0.663409,0.178551,0.090909,1.00,0.667328,0.661917,0.682761,0.651961,0.110601,-0.150667,0.185920,0.162012,0.106240,48.936269,2.358867,22.0
2,11104.0,0.168161,0.141766,0.000000,0.95,0.166947,0.168622,0.117688,0.198200,0.189231,-0.050379,-0.160599,-0.064248,-0.022274,48.855908,2.392571,20.0
3,5110.0,0.220518,0.181978,0.000000,1.00,0.186879,0.233298,0.207737,0.228106,0.235257,0.037136,0.006019,0.108280,-0.132022,48.843893,2.351966,23.0
4,6108.0,0.469135,0.247331,0.000000,1.00,0.627981,0.409138,0.422837,0.496600,0.145883,-0.308258,-0.146049,0.080851,0.099821,48.847082,2.321375,17.0


# CLEAN FEATURE TABLE

In [7]:
station_features = station_features.replace([np.inf, -np.inf], np.nan)
station_features = station_features.dropna()

print("After cleaning:", station_features.shape)

After cleaning: (1474, 17)


# SAVE STATION FEATURES

In [8]:
station_features.to_csv(f"{STATION_FEATURE_FOLDER}/station_features.csv", index=False)
print("✅ Saved station_features.csv")

✅ Saved station_features.csv
